In [ ]:
import torch
import torchvision
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.utils.data as data
import wandb
import os

In [ ]:
!pip install lightly

In [ ]:
from lightly.loss import NTXentLoss
from lightly.models.modules import SimCLRProjectionHead
from lightly.transforms.simclr_transform import SimCLRTransform

In [ ]:
if os.path.exists("/kaggle/working"):
    base_path = "/kaggle/working"

elif "google.colab" in str(get_ipython()):
    base_path = "/content/drive/MyDrive/ProjectSimCLR"

else:
    base_path = "./"

models_path = os.path.join(base_path, 'models')

os.makedirs(models_path, exist_ok=True)

def save_checkpoint(model, optimizer, epoch, loss, run_id):

    torch.save(
        {
            "epoch": epoch,
            "optimizer": optimizer.state_dict(),
            "model": model.state_dict(),
            "loss": loss,
            "wandb_run_id": run_id,
        },
        os.path.join(models_path, f"{run_id}_epoch_{epoch}.pth"),
    )


def load_checkpoint(model, optimizer, path, map_location=None):
    """Loads checkpoints produced by save_checkpoint (pickled model and optimizer objects)."""
    if map_location is None:
        if torch.cuda.is_available():
            map_location = "cuda"
        elif torch.backends.mps.is_available():
            map_location = "mps"
        else:
            map_location = "cpu"
    checkpoint = torch.load(path, weights_only=False, map_location=map_location)
    saved_model = checkpoint["model"]
    saved_optimizer = checkpoint["optimizer"]
    model.load_state_dict(saved_model.state_dict())
    optimizer.load_state_dict(saved_optimizer.state_dict())
    return checkpoint["epoch"], checkpoint["loss"], checkpoint["wandb_run_id"]

In [ ]:
class SimCLR(nn.Module):
    def __init__(self, encoder, projection_head):
        super().__init__()
        self.encoder = encoder
        self.projection_head = projection_head

    def forward(self, x):
        x = self.encoder(x)
        x = torch.flatten(x, 1)
        x = self.projection_head(x)
        return x


In [ ]:
resnet = torchvision.models.resnet18()
backbone = nn.Sequential(*list(resnet.children())[:-1])
projection_head = SimCLRProjectionHead(input_dim=512, hidden_dim=512, output_dim=128)
model = SimCLR(backbone, projection_head)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model.to(device)

In [ ]:
# Uncomment for running on kaggle
"""
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_KEY")
wandb.login(key=wandb_key)
"""

In [ ]:
hp = {
    "batch_size": 256,
    "pipeline_test": True,
    "epochs": 50,
    "temperature": 0.2, 
    "optim": {"base_lr": 0.06, "weight_decay": 1e-5},
}

In [ ]:
transform = SimCLRTransform(input_size=32, gaussian_blur=0.0)
dataset = datasets.CIFAR10(
    root="./data", train=True, transform=transform, download=True
)
if hp["pipeline_test"]:
    dataset = data.Subset(dataset, range(256))

In [ ]:
dataloader = DataLoader(
    dataset, batch_size=hp["batch_size"], shuffle=True, drop_last=True, num_workers=2
)

In [ ]:
criterion = NTXentLoss(temperature=hp["temperature"])
optimizer = torch.optim.SGD(params=model.named_parameters(), lr=hp["optim"]["base_lr"] * hp["batch_size"] / 256, weight_decay=hp["optim"]["weight_decay"])
# Uncomment when using multiple GPUs
model = torch.nn.DataParallel(model)

In [ ]:
print("Starting Training ...")
run = wandb.init(project="simclr-cifar10", config=hp, mode='online')
model = model.to(device)
for epoch in range(hp["epochs"]):
    print(f"Epoch {epoch} of {hp['epochs']}")
    total_loss = 0
    for batch in dataloader:
        x0, x1 = batch[0]
        x0 = x0.to(device)
        x1 = x1.to(device)
        z0 = model(x0)
        z1 = model(x1)

        loss = criterion(z0, z1)
        total_loss += loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    avg_loss = total_loss / len(dataloader)
    wandb.log({"loss": avg_loss, "epoch": epoch, "lr": optimizer.param_groups[0]["lr"]})
    if epoch % 10 == 0 and epoch >= 20:
        save_checkpoint(model, optimizer, epoch, loss, run.id)
    print(f"Epoch: {epoch}, Avg_loss: {avg_loss:.5f}")

In [ ]:
if torch.cuda.is_available():
    torch_device = torch.device("cuda")
elif torch.backends.mps.is_available():
    torch_device = torch.device("mps")
else:
    torch_device = torch.device("cpu")

resnet = torchvision.models.resnet18()
backbone = nn.Sequential(*list(resnet.children())[:-1])
projection_head = SimCLRProjectionHead(input_dim=512, hidden_dim=512, output_dim=128)
model = SimCLR(backbone, projection_head)
optimizer = torch.optim.SGD(params=model.named_parameters(), **hp["optim"])
epoch, loss, run_id = load_checkpoint(
    model,
    optimizer,
    "/Users/hannesneumann/Repositories/SelfSupervisedLearning/models/kxv5e225_epoch_20.pth",
    map_location=torch_device,
)
model.to(torch_device)
print(model)

In [ ]:
def linear_evaluation(backbone, train_loader, test_loader, device):
    # Encoder einfrieren
    for param in backbone.parameters():
        param.requires_grad = False

    classifier = nn.Linear(512, 10).to(device)  # 10 CIFAR-10 Klassen
    optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
    # Nur Classifier trainieren
    total_loss = 0
    for epoch in range(20):
        for imgs, labels in train_loader:
            features = backbone(imgs.to(device))
            preds = classifier(features)
            loss = nn.CrossEntropyLoss()(preds, labels.to(device))
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss
        
    
    # Evaluate the models performance on the test set
    model.eval()
    classifier.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            features = backbone(imgs.to(device))
            preds = classifier(features)
            predicted = preds.argmax(dim=1)
            correct += (predicted == labels.to(device)).sum().item()
            total += labels.size(0)
    accuracy = correct / total if total > 0 else 0


    wandb.log({"linear_eval_accuracy": accuracy})

In [ ]:
# test_set =  datasets.CIFAR10(root="./data", train=False, download=True)
# test_looader = DataLoader(test_set)
# linear_evaluation(model.encoder, dataloader, test_looader, device)
